# 🎬 Khám Phá và Phân Tích Dữ Liệu Netflix (EDA)
### **Nhóm 8 - KADA**
**Thành viên:** Phạm Nguyễn Hải Triều, Lương Võ Khôi Quốc, Mô Ha Mách Bu Ba Ka, Trần Quốc Huy, Nguyễn Chí Thịnh

Dự án sử dụng các thư viện Data Science cốt lõi:
- **Pandas & NumPy:** Xử lý và tiền xử lý dữ liệu
- **Matplotlib & Seaborn:** Trực quan hóa dữ liệu trực quan
- **Scikit-learn:** Khai phá đặc trưng (TF-IDF) và xây dựng hệ thống gợi ý nội dung (Content-Based Recommendation)

## 1. Import Thư Viện và Cấu Hình Giao Diện Biểu Đồ

In [ ]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Thiết lập phong cách hiển thị
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('tab10')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

## 2. Đọc và Khảo Sát Tổng Quan Tập Dữ Liệu

In [ ]:
df = pd.read_csv('netflix_titles.csv')
print(f"Kích thước dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột\n")
df.head()

In [ ]:
# Xem thông tin kiểu dữ liệu và giá trị thiếu
df.info()

In [ ]:
# Thống kê tỷ lệ giá trị thiếu (Missing Values)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Số lượng thiếu': missing, 'Tỷ lệ (%)': missing_pct.round(2)})
missing_df[missing_df['Số lượng thiếu'] > 0]

## 3. Tiền Xử Lý Dữ Liệu (Data Cleaning & Transformation)

In [ ]:
df_clean = df.copy()

# 1. Điền giá trị trống cho các cột phân loại
df_clean['director'] = df_clean['director'].fillna('Unknown')
df_clean['cast'] = df_clean['cast'].fillna('Unknown')
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['rating'] = df_clean['rating'].fillna(df_clean['rating'].mode()[0])

# 2. Chuyển đổi date_added thành datetime để phân tích thời gian
df_clean['date_added'] = df_clean['date_added'].str.strip()
df_clean['date_added'] = pd.to_datetime(df_clean['date_added'], format='%B %d, %Y', errors='coerce')
df_clean['year_added'] = df_clean['date_added'].dt.year
df_clean['month_added'] = df_clean['date_added'].dt.month_name()

# 3. Trích xuất thời lượng dưới dạng số (duration_num)
df_clean['duration_num'] = df_clean['duration'].str.extract('(\d+)').astype(float)
df_clean['duration_unit'] = df_clean['duration'].str.extract('([a-zA-Z]+)')

print("Hoàn tất làm sạch dữ liệu!")
df_clean[['title', 'type', 'country', 'year_added', 'duration_num', 'duration_unit']].head()

## 4. Trực Quan Hóa và Phân Tích Chuyên Sâu (Visualizations)

### 4.1 Tỷ Lệ Phim Lẻ (Movie) và Phim Bộ (TV Show)

In [ ]:
plt.figure(figsize=(7, 7))
type_counts = df_clean['type'].value_counts()
colors = ['#e50914', '#221f1f']
plt.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', startangle=140, colors=colors,
        explode=[0.05, 0], shadow=True, textprops={'fontsize': 12, 'weight': 'bold'})
plt.title('Tỷ Lệ Movie vs TV Show Trên Netflix', fontsize=14, weight='bold')
plt.show()

### 4.2 Xu Hướng Thêm Phim Mới Qua Các Năm

In [ ]:
plt.figure(figsize=(12, 5))
yearly_trend = df_clean.groupby(['year_added', 'type']).size().unstack().fillna(0)
yearly_trend.plot(kind='line', marker='o', linewidth=2.5, color=['#e50914', '#0071eb'], ax=plt.gca())
plt.title('Xu Hướng Tăng Trưởng Nội Dung Theo Năm', fontsize=14, weight='bold')
plt.xlabel('Năm', fontsize=12)
plt.ylabel('Số Lượng Phim', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### 4.3 Top 10 Quốc Gia Sản Xuất Nhiều Nội Dung Nhất

In [ ]:
plt.figure(figsize=(12, 6))
top_countries = df_clean[df_clean['country'] != 'Unknown']['country'].str.split(', ').explode().value_counts().head(10)
sns.barplot(x=top_countries.values, y=top_countries.index, palette='viridis', hue=top_countries.index, legend=False)
plt.title('Top 10 Quốc Gia Sản Xuất Nội Dung Nhiều Nhất', fontsize=14, weight='bold')
plt.xlabel('Số Lượng Tác Phẩm', fontsize=12)
plt.ylabel('Quốc Gia', fontsize=12)
for i, v in enumerate(top_countries.values):
    plt.text(v + 15, i, str(v), color='black', va='center', fontweight='bold')
plt.show()

### 4.4 Phân Bổ Độ Tuổi (Rating)

In [ ]:
plt.figure(figsize=(12, 6))
rating_order = df_clean['rating'].value_counts().index
sns.countplot(data=df_clean, x='rating', order=rating_order, hue='type', palette={'Movie': '#e50914', 'TV Show': '#221f1f'})
plt.title('Phân Bố Xếp Hạng Độ Tuổi Theo Loại Nội Dung', fontsize=14, weight='bold')
plt.xlabel('Xếp Hạng (Rating)', fontsize=12)
plt.ylabel('Số Lượng', fontsize=12)
plt.xticks(rotation=45)
plt.show()

### 4.5 Phân Bố Thời Lượng Phim Lẻ (Movie Duration)

In [ ]:
plt.figure(figsize=(10, 5))
movies_df = df_clean[df_clean['type'] == 'Movie']
sns.histplot(movies_df['duration_num'], kde=True, bins=35, color='#e50914')
mean_val = movies_df['duration_num'].mean()
plt.axvline(mean_val, color='blue', linestyle='dashed', linewidth=2, label=f'Thời lượng TB: {mean_val:.1f} phút')
plt.title('Phân Bố Thời Lượng Phim Lẻ (Phút)', fontsize=14, weight='bold')
plt.xlabel('Thời Lượng (Phút)', fontsize=12)
plt.ylabel('Tần Suất', fontsize=12)
plt.legend()
plt.show()

### 4.6 Top 10 Thể Loại Phổ Biến Nhất

In [ ]:
plt.figure(figsize=(12, 6))
top_genres = df_clean['listed_in'].str.split(', ').explode().value_counts().head(10)
sns.barplot(x=top_genres.values, y=top_genres.index, palette='mako', hue=top_genres.index, legend=False)
plt.title('Top 10 Thể Loại Phim Phổ Biến Nhất', fontsize=14, weight='bold')
plt.xlabel('Số Lượng', fontsize=12)
plt.ylabel('Thể Loại', fontsize=12)
for i, v in enumerate(top_genres.values):
    plt.text(v + 10, i, str(v), color='black', va='center', fontweight='bold')
plt.show()

## 5. Ứng Dụng Machine Learning: Hệ Thống Gợi Ý Phim Dựa Trên Nội Dung (TF-IDF & Cosine Similarity)

In [ ]:
# Kết hợp thông tin thể loại và tóm tắt nội dung
df_clean['features'] = df_clean['listed_in'] + ' ' + df_clean['description']

# Trích xuất đặc trưng văn bản bằng TF-IDF
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df_clean['features'].fillna(''))

# Tính toán ma trận độ tương đồng Cosine
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Kích thước ma trận tương đồng Cosine: {cosine_sim.shape}")

In [ ]:
def recommend_movies(title, num_recommendations=5):
    matches = df_clean[df_clean['title'].str.lower() == title.lower()]
    if matches.empty:
        return f"Không tìm thấy phim '{title}'!"
    
    idx = matches.index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:num_recommendations+1]
    
    movie_indices = [i[0] for i in sim_scores]
    return df_clean.iloc[movie_indices][['title', 'type', 'listed_in', 'release_year', 'rating']]

# Thử nghiệm gợi ý cho phim "Squid Game"
print("Gợi ý các phim tương tự 'Squid Game':")
recommend_movies('Squid Game')

In [ ]:
# Thử nghiệm gợi ý cho phim phiêu lưu / khoa học viễn tưởng "Dark Skies"
print("Gợi ý các phim tương tự 'Dark Skies':")
recommend_movies('Dark Skies')